In [7]:
import seaborn as sns
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

# Load data
df = sns.load_dataset("titanic")

# Drop unnecessary columns
df.drop(["class","who","adult_male","deck","embark_town","alive"],
        axis=1, inplace=True)

# Split
X = df.drop(columns=["survived"])
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Column types
numeric_features = ["pclass", "age", "sibsp", "parch", "fare"]
categorical_features = ["sex", "embarked", "alone"]

results = []

# ============================
# Model Configurations
# ============================

models_config = {
    "Logistic Regression": {
        "model": LogisticRegression(max_iter=1000),
        "scaling": True,
        "drop_first": True
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "scaling": True,
        "drop_first": False
    },
    "Naive Bayes": {
        "model": GaussianNB(),
        "scaling": False,
        "drop_first": False
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(),
        "scaling": False,
        "drop_first": False
    },
    "SVM": {
        "model": SVC(),
        "scaling": True,
        "drop_first": True
    }
}

# ============================
# Training Loop
# ============================
best_score = 0
best_model = None
for name, config in models_config.items():

    # Numeric pipeline
    if config["scaling"]:
        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
    else:
        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])

    # Categorical pipeline
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            drop="first" if config["drop_first"] else None,
            handle_unknown="ignore"
        ))
    ])

    # ColumnTransformer
    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ])

    pipe = Pipeline([
        ("preprocessing", preprocessor),
        ("model", config["model"])
    ])

    # Train
    pipe.fit(X_train, y_train)

    # Predict
    y_pred = pipe.predict(X_test)

    # Metrics
    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4)
    })
    if accuracy_score > best_score:
        best_score = accuracy_score
        best_model = pipe

results_df = pd.DataFrame(results)
results_df

TypeError: '>' not supported between instances of 'function' and 'int'

In [3]:
from sklearn.model_selection import cross_val_score

svm_cv = cross_val_score(best_svm_pipeline, X, y, cv=5)

print("SVM CV Mean:", svm_cv.mean())
print("SVM CV Std:", svm_cv.std())

NameError: name 'best_svm_pipeline' is not defined